# 20 - vLLM latest baseline (its own notebook: it installs its own torch)

In [ ]:
!pip install -q vllm
!python -c "import vllm, torch; print(vllm.__version__, torch.__version__)"
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-1.5B-Instruct

In [ ]:
import subprocess, time, requests
log = open("/kaggle/working/vllm.log", "w")
p = subprocess.Popen(["vllm", "serve", "/kaggle/working/models/Qwen2.5-1.5B-Instruct", "--served-model-name", "qwen",
                      "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.85",
                      "--port", "8001"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get("http://localhost:8001/health", timeout=2).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(5)
print(requests.post("http://localhost:8001/v1/completions",
                    json={"model": "qwen", "prompt": "Hello", "max_tokens": 16}).json())

In [ ]:
!grep -i -E "backend|attention" /kaggle/working/vllm.log | head -20

Record the vLLM version, torch version, the attention backend and whether it works in `docs/baselines.md`. Then clone the repo, `pip install httpx transformers`, and run the SAME `serve_bench` commands against port 8001 (and the prefill set with `--max-model-len 8192`).